# Hierarchical Prototype Matching Network (HPMN)

## Idea

This model replaces ordinary dense hidden units with **prototype matchers**.

Instead of only learning a weighted sum of features, each hidden unit learns:

- a prototype center,
- which features matter for that prototype,
- how strict the match should be,
- several ways of measuring similarity,
- and how strongly each similarity should contribute.

The final architecture is:

```text
fold-safe engineered features
(204 features in the historical Fold 1)
        |
        v
256 prototype matchers
        |
        v
conditional routing
        |
        v
128 prototype matchers
        |
        v
linear readout from both matching levels
```

The second matching layer therefore learns **patterns of first-layer matches**.

---

## 1. Prototype matching

Each matcher has a learned center, similar to a reference point.

For an input sample, the matcher asks:

> How similar is this sample to my prototype?

But it does not have to use every feature.

Each matcher learns a soft feature mask, so some features can matter a lot while others matter almost nothing.

The total amount of active feature weight is called:

```text
M_eff
```

You can think of `M_eff` as the approximate number of features that a matcher is currently using.

Different matchers therefore become specialists in different parts of the feature space.

---

## 2. Learnable matching strictness

The feature mask uses a temperature parameter.

High temperature:

```text
softer feature selection
```

Low temperature:

```text
sharper feature selection
```

There is a global temperature schedule during training, but each matcher can also learn a small relative adjustment.

This lets one matcher become more selective while another remains more tolerant.

The relative adjustments are centered inside each layer, so the entire layer cannot simply ignore the global schedule.

---

## 3. Four similarity geometries

Each matcher compares the sample with its prototype in four different ways.

### Euclidean similarity

This measures ordinary distance.

Small distance means a strong match.

The distance is normalized by `M_eff`, so a matcher is not punished just because it uses more features.

---

### Cosine similarity

Cosine similarity asks whether the sample and prototype point in a similar direction.

This is useful because it focuses more on the **shape** of the feature vector and less on its absolute magnitude.

---

### Signed mismatch pattern

The model also looks directly at:

```text
sample - prototype
```

This difference vector is passed through a tiny matcher-specific neural network.

That branch can learn patterns such as:

```text
feature A is above the prototype
feature B is below the prototype
feature C is roughly equal
```

Two samples can have the same total distance from a prototype but still have very different mismatch patterns.

---

### Dot4: normalized weighted dot product

The fourth geometry is a weighted dot product.

Conceptually:

```text
weighted dot(sample, prototype) / M_eff
```

This keeps some magnitude-sensitive information that cosine similarity removes.

It is passed through a sigmoid so the final score stays bounded.

---

## 4. Learned response curves

A raw similarity score does not always have to be interpreted linearly.

For example:

```text
cosine = 0.60  -> maybe weak evidence
cosine = 0.80  -> maybe medium evidence
cosine = 0.95  -> maybe very strong evidence
```

Each matcher therefore learns a small monotonic response curve for every geometry.

So the model learns not only:

> Which geometry is useful?

but also:

> How should that geometry be interpreted?

---

## 5. Learned geometry mixture

After the four similarity scores are calibrated, the matcher combines them with learned weights.

A matcher might end up behaving roughly like:

```text
Euclidean : 20%
Cosine    : 20%
Pattern   : 40%
Dot4      : 20%
```

Another matcher can learn a completely different mixture.

This allows different prototype units to specialize in different notions of similarity.

---

## 6. Hierarchical matching

The first layer creates 256 prototype-matching responses:

```text
input -> h1
```

The second layer then matches patterns inside those first-layer responses:

```text
h1 -> h2
```

So the second layer is not simply matching raw features again.

It is matching **patterns of matches**.

That is the main hierarchical idea behind HPMN.

---

## 7. Conditional routing

Before `h1` is sent into the second matching layer, the model creates a sample-dependent routing vector.

The router looks at the current first-layer response and decides:

```text
focus more on these first-layer matchers
focus less on these others
```

For example:

```text
routing weights = [1.4, 0.6, 1.0, 1.7, ...]
```

Where:

```text
> 1.0  = emphasize
< 1.0  = reduce
≈ 1.0  = leave mostly unchanged
```

The routed representation is then:

```text
routed_h1 = h1 * routing_weights
```

and this routed version is passed into the second prototype-matching layer.

The routing weights are normalized so their mean is approximately 1 for each sample.

This prevents the router from helping only by globally increasing or decreasing the scale.

The router is also initialized as an identity transform:

```text
all routing weights start at 1
```

So conditional behavior is only learned if it actually improves the objective.

---

## 8. Linear skip readout

The final prediction uses both matching levels:

```text
[h1, h2] -> linear output
```

This is useful because some information may already be easy to detect in the first matching layer.

The model does not have to force every useful signal through the second layer.

---

## 9. Feature representation

The input representation is rebuilt independently inside every outer fold.

It contains:

- 7 raw numerical features
- 56 decimal-digit features
- 4 domain flags
- 69 frequency encodings
- 69 target encodings with `smooth="auto"`
- 69 target encodings with `smooth=10`
- 6 raw categorical IDs

Constant numerical columns are removed using only the training rows of the current fold.

In the historical Fold 1 this produces:

```text
198 numerical features
+ 6 categorical IDs
= 204 final features
```

The target encodings are created out-of-fold inside the training partition.

Validation and competition-test rows are never used to fit those encoders.

---

## 10. Training and validation

The final model uses stratified 5-fold cross-validation.

For every fold:

1. rebuild the feature pipeline from that fold's training rows,
2. train the model,
3. monitor validation ROC AUC,
4. keep the best checkpoint,
5. restore the best model before inference,
6. predict both validation rows and the Kaggle test set.

One important implementation detail is that the mask temperature is stored as a normal Python attribute rather than inside the PyTorch `state_dict`.

Therefore the best mask temperature is saved separately and restored together with the best model weights.

---

## 11. Kaggle submission

Each fold produces one probability prediction for every Kaggle test row.

The final prediction is simply:

```text
final prediction
    =
average of the 5 fold predictions
```

The script then writes:

```text
submission.csv
```

It also saves full out-of-fold predictions so the complete 5-fold ROC AUC can be measured locally.

---

## Short intuition

The model can be summarized as:

> Learn prototype patterns, learn which features matter for each pattern, compare samples using several similarity measures, conditionally refocus the first matching layer, and then match the resulting pattern of matches again.

The goal is to get some of the local and conditional behavior of tree models while keeping the whole architecture differentiable and prototype-based.


---

## Implementation note

The final training script intentionally keeps only the components needed for the selected architecture and for reproducible 5-fold training.

It does **not** include the experimental 3-layer hierarchy, hard parent-child routing, or nested pattern-cosine branch that were tested separately.

The saved per-fold artifacts are limited to useful training history, validation predictions, model checkpoints, the final OOF predictions, a fold summary, and `submission.csv`.


In [ ]:
"""
Hierarchical Prototype Matching Network (HPMN)
Final 5-fold Kaggle training script for Playground Series S6E9.

Architecture
------------
fold-safe engineered features (204 in the historical Fold 1)
    -> 256 prototype matchers
    -> sample-dependent conditional routing
    -> 128 hierarchical prototype matchers
    -> linear readout from [h1, h2]

Each prototype matcher learns:
- a prototype center,
- a soft feature subset,
- a relative local mask temperature,
- Euclidean, cosine, signed-pattern, and normalized-dot similarities,
- one monotonic calibration curve per similarity geometry,
- and a learned mixture of those calibrated geometries.

The script performs stratified 5-fold cross-validation, restores the true best
checkpoint (including the non-state_dict mask temperature), creates OOF
predictions, averages the five test-probability vectors, and writes a Kaggle
submission file.
"""

import os
import gc
import glob
import math
import copy
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------------------
# Reproducibility and competition configuration
# ------------------------------------------------------------------------------------------

SEED = 21
ID = "id"
TARGET = "Will_Buy_EV"
N_SPLITS = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RAW_NUMS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

RAW_CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

DIGIT_KS = list(range(-4, 4))

# ------------------------------------------------------------------------------------------
# Matching-network configuration
# ------------------------------------------------------------------------------------------

N_MATCH1 = 256
N_MATCH2 = 128
PATTERN_HIDDEN = 8

# Low-rank sample-dependent routing between h1 and h2.
# The route is initialized to identity and normalized per sample so it cannot
# improve merely by globally rescaling the entire first-layer representation.
ROUTING_RANK = 12
LAMBDA_ROUTING = 1e-6
N_FIRE_KNOTS = 8
GEOMETRY_MIX_TEMP = 1.0

INIT_TARGET_M1 = 20.0
INIT_TARGET_M2 = 30.0

MASK_TEMP_START = 0.90
MASK_TEMP_END = 0.30

# Local learnable temperature per matcher.
# Each matcher learns a RELATIVE multiplicative offset around the global schedule:
#
#     alpha_centered_i = alpha_i - mean(alpha)
#     T_i(t) = clip(T_global(t) * exp(alpha_centered_i), T_MIN, T_MAX)
#
# alpha_i starts at 0, so the model initially matches the baseline schedule.
# Centering prevents the entire layer from drifting globally softer/harder.
LOCAL_T_MIN = 0.03
LOCAL_T_MAX = 1.50
LOCAL_T_LOG_SCALE_INIT_STD = 0.00

# Weak regularization keeps local temperatures from drifting excessively
# unless validation loss provides a reason to do so.
LAMBDA_LOCAL_T = 1e-6

# Weak diversity regularization between normalized feature gates.
LAMBDA_GATE_DIVERSITY = 2e-6

MAX_EPOCHS = 40
PATIENCE = 9
BATCH_SIZE = 2048
PRED_BATCH_SIZE = 8192
LR = 1.5e-3
WEIGHT_DECAY = 2e-5
GRAD_CLIP = 5.0

LAMBDA_MASK_SIZE = 2e-6
LAMBDA_MASK_BINARY = 1e-6
LAMBDA_FIRE_SMOOTH = 1e-5
LAMBDA_HEAD_L2 = 1e-5

CENTER_INIT_ROWS = 8192

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------------------
# Load competition data
# ------------------------------------------------------------------------------------------

competition_roots = [
    "/kaggle/input/competitions/playground-series-s6e9",
    "/kaggle/input/playground-series-s6e9",
]

train_path = None
test_path = None
sample_path = None

for root in competition_roots:
    candidate_train = os.path.join(root, "train.csv")
    candidate_test = os.path.join(root, "test.csv")
    candidate_sample = os.path.join(root, "sample_submission.csv")

    if os.path.exists(candidate_train) and os.path.exists(candidate_test):
        train_path = candidate_train
        test_path = candidate_test
        sample_path = candidate_sample if os.path.exists(candidate_sample) else None
        break

if train_path is None:
    candidates = [
        p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
        if "playground-series-s6e9" in p.lower()
    ]
    if not candidates:
        raise FileNotFoundError("Could not locate the S6E9 competition files.")

    train_path = candidates[0]
    root = os.path.dirname(train_path)
    test_path = os.path.join(root, "test.csv")
    sample_path = os.path.join(root, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Convert the target robustly whether Kaggle stores it as Yes/No or already as 0/1.
if not pd.api.types.is_numeric_dtype(train[TARGET]):
    train[TARGET] = (
        train[TARGET]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0.0,
            "yes": 1.0,
            "0": 0.0,
            "1": 1.0,
            "false": 0.0,
            "true": 1.0,
        })
    )
else:
    train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")

if train[TARGET].isna().any():
    raise ValueError("Target conversion produced missing values.")

train[TARGET] = train[TARGET].astype(np.float32)
y = train[TARGET].to_numpy(dtype=np.float32)

print("Train:", train.shape)
print("Test :", test.shape)
print("Positive rate:", float(y.mean()))


# ## Fold-safe feature engineering
# 
# The historical record representation is rebuilt independently inside every outer fold.

# ==========================================================================================
# Fold-safe record feature engineering
# ==========================================================================================

def safe_numeric(series):
    """Convert a column to float while preserving missing values."""
    return pd.to_numeric(series, errors="coerce").astype(np.float64)


def stringify_values(values):
    """Convert arbitrary values into stable categorical strings."""
    s = pd.Series(values, copy=False).astype("object")
    s = s.where(~pd.isna(s), "__NA__")
    return s.astype(str).to_numpy(dtype=object)


def build_base_frame(df):
    """
    Build the 67 base numerical features:
      - 7 raw numerics
      - 56 extracted decimal digits
      - 4 domain flags
    """
    out = pd.DataFrame(index=np.arange(len(df)))

    for col in RAW_NUMS:
        out[col] = safe_numeric(df[col]).to_numpy()

    for col in RAW_NUMS:
        x = safe_numeric(df[col]).to_numpy()

        for k in DIGIT_KS:
            scale = 10.0 ** k
            digit = np.floor(x / scale)
            digit = np.mod(digit, 10.0)
            out[f"{col}__digit_k{k}"] = digit

    income = safe_numeric(df["Annual_Income_USD"]).to_numpy()
    concern = safe_numeric(df["Environmental_Concern_Level"]).to_numpy()

    out["flag_income_eq_30000"] = (income == 30000).astype(np.float64)
    out["flag_income_ge_170537"] = (income >= 170537).astype(np.float64)
    out["flag_income_38k_42k"] = (
        (income >= 38000) & (income <= 42000)
    ).astype(np.float64)
    out["flag_env_concern_eq_1"] = (concern == 1).astype(np.float64)

    return out


def fill_base_from_train(train_base, valid_base, test_base):
    """Median-impute base features using outer-fold training rows only."""
    for col in train_base.columns:
        median = np.nanmedian(train_base[col].to_numpy(dtype=np.float64))

        if not np.isfinite(median):
            median = 0.0

        train_base[col] = train_base[col].fillna(median)
        valid_base[col] = valid_base[col].fillna(median)
        test_base[col] = test_base[col].fillna(median)


def source_values(raw_df, base_df, source_name):
    """Return one of the 69 categorical sources used for frequency/target encoding."""
    kind, col = source_name.split("::", 1)

    if kind == "rawcat":
        return stringify_values(raw_df[col].to_numpy())

    if kind == "numcat":
        return stringify_values(safe_numeric(raw_df[col]).to_numpy())

    if kind == "digitcat":
        return stringify_values(base_df[col].to_numpy())

    raise ValueError(source_name)


def make_single_col_matrix(values):
    return np.asarray(values, dtype=object).reshape(-1, 1)


def target_encode_builtin(
    train_values,
    valid_values,
    test_values,
    y_train,
    smooth,
    seed,
):
    """
    Fold-safe sklearn TargetEncoder.

    fit_transform() produces OOF values for the current outer-fold training rows.
    transform() is then applied to outer validation and competition test rows.
    """
    encoder = TargetEncoder(
        smooth=smooth,
        cv=5,
        shuffle=True,
        random_state=seed,
        target_type="binary",
    )

    train_encoded = encoder.fit_transform(
        make_single_col_matrix(train_values),
        y_train,
    ).reshape(-1)

    valid_encoded = encoder.transform(
        make_single_col_matrix(valid_values)
    ).reshape(-1)

    test_encoded = encoder.transform(
        make_single_col_matrix(test_values)
    ).reshape(-1)

    return train_encoded, valid_encoded, test_encoded


def build_fold_features(train_raw, test_raw, train_idx, valid_idx, fold_number):
    """
    Rebuild the complete record representation independently for one outer fold.

    Expected Fold-1 accounting:
        67 base
      + 69 frequency
      + 69 TE(auto)
      + 69 TE(10)
      = 274 numerical candidates

    Historical constant removal:
        274 - 76 = 198 numerical features

    Plus 6 raw categorical IDs:
        198 + 6 = 204 final features
    """
    fold_train = train_raw.iloc[train_idx].reset_index(drop=True)
    fold_valid = train_raw.iloc[valid_idx].reset_index(drop=True)

    y_train = fold_train[TARGET].to_numpy(dtype=np.float32)
    y_valid = fold_valid[TARGET].to_numpy(dtype=np.float32)

    base_train = build_base_frame(fold_train)
    base_valid = build_base_frame(fold_valid)
    base_test = build_base_frame(test_raw)

    fill_base_from_train(base_train, base_valid, base_test)

    digit_cols = [
        col for col in base_train.columns
        if "__digit_k" in col
    ]

    encoding_sources = (
        [f"rawcat::{col}" for col in RAW_CATS]
        + [f"numcat::{col}" for col in RAW_NUMS]
        + [f"digitcat::{col}" for col in digit_cols]
    )

    assert len(encoding_sources) == 69

    # ------------------------------------------------------------------
    # Frequency encoding
    # ------------------------------------------------------------------

    freq_train = pd.DataFrame(index=np.arange(len(fold_train)))
    freq_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    freq_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for source in encoding_sources:
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        frequency_map = (
            pd.Series(train_values)
            .value_counts(dropna=False, normalize=True)
            .to_dict()
        )

        name = f"freq::{source}"

        freq_train[name] = (
            pd.Series(train_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_valid[name] = (
            pd.Series(valid_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_test[name] = (
            pd.Series(test_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

    # ------------------------------------------------------------------
    # Raw categorical IDs
    # ------------------------------------------------------------------

    catid_train = pd.DataFrame(index=np.arange(len(fold_train)))
    catid_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    catid_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for col in RAW_CATS:
        train_values = stringify_values(fold_train[col].to_numpy())
        valid_values = stringify_values(fold_valid[col].to_numpy())
        test_values = stringify_values(test_raw[col].to_numpy())

        unique_values = pd.Index(pd.unique(train_values))
        mapping = {
            value: i + 1
            for i, value in enumerate(unique_values)
        }

        name = f"catid::{col}"

        catid_train[name] = (
            pd.Series(train_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_valid[name] = (
            pd.Series(valid_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_test[name] = (
            pd.Series(test_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

    # ------------------------------------------------------------------
    # Two historical target-encoding blocks
    # ------------------------------------------------------------------

    te_auto_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_auto_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_auto_test = pd.DataFrame(index=np.arange(len(test_raw)))

    te_10_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_10_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_10_test = pd.DataFrame(index=np.arange(len(test_raw)))

    # Use a deterministic encoder seed for each outer fold.
    encoder_seed = SEED + (fold_number - 1)

    print(f"Fold {fold_number}: building 2 x 69 target encodings...")

    for j, source in enumerate(encoding_sources):
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        auto_train, auto_valid, auto_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth="auto",
            seed=encoder_seed,
        )

        ten_train, ten_valid, ten_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth=10.0,
            seed=encoder_seed,
        )

        te_auto_train[f"te_auto::{source}"] = auto_train
        te_auto_valid[f"te_auto::{source}"] = auto_valid
        te_auto_test[f"te_auto::{source}"] = auto_test

        te_10_train[f"te_10::{source}"] = ten_train
        te_10_valid[f"te_10::{source}"] = ten_valid
        te_10_test[f"te_10::{source}"] = ten_test

        if (j + 1) % 10 == 0 or j + 1 == len(encoding_sources):
            print(f"  encoded {j + 1}/{len(encoding_sources)} sources")

    numeric_train = pd.concat(
        [
            base_train.reset_index(drop=True),
            freq_train,
            te_auto_train,
            te_10_train,
        ],
        axis=1,
    )

    numeric_valid = pd.concat(
        [
            base_valid.reset_index(drop=True),
            freq_valid,
            te_auto_valid,
            te_10_valid,
        ],
        axis=1,
    )

    numeric_test = pd.concat(
        [
            base_test.reset_index(drop=True),
            freq_test,
            te_auto_test,
            te_10_test,
        ],
        axis=1,
    )

    assert numeric_train.shape[1] == 274, numeric_train.shape

    numeric_train = numeric_train.replace([np.inf, -np.inf], np.nan)
    numeric_valid = numeric_valid.replace([np.inf, -np.inf], np.nan)
    numeric_test = numeric_test.replace([np.inf, -np.inf], np.nan)

    # Final train-only median imputation.
    for col in numeric_train.columns:
        median = np.nanmedian(
            numeric_train[col].to_numpy(dtype=np.float64)
        )

        if not np.isfinite(median):
            median = 0.0

        numeric_train[col] = numeric_train[col].fillna(median)
        numeric_valid[col] = numeric_valid[col].fillna(median)
        numeric_test[col] = numeric_test[col].fillna(median)

    # Remove numerical columns that are constant in this fold's training rows.
    unique_counts = numeric_train.nunique(dropna=False)

    keep_numeric_cols = unique_counts[
        unique_counts > 1
    ].index.tolist()

    dropped_numeric_cols = unique_counts[
        unique_counts <= 1
    ].index.tolist()

    numeric_train = numeric_train[keep_numeric_cols]
    numeric_valid = numeric_valid[keep_numeric_cols]
    numeric_test = numeric_test[keep_numeric_cols]

    full_train = pd.concat(
        [
            numeric_train.reset_index(drop=True),
            catid_train,
        ],
        axis=1,
    )

    full_valid = pd.concat(
        [
            numeric_valid.reset_index(drop=True),
            catid_valid,
        ],
        axis=1,
    )

    full_test = pd.concat(
        [
            numeric_test.reset_index(drop=True),
            catid_test,
        ],
        axis=1,
    )

    feature_names = full_train.columns.tolist()

    print(
        f"Fold {fold_number}: "
        f"{len(keep_numeric_cols)} numerical + "
        f"{catid_train.shape[1]} categorical IDs = "
        f"{len(feature_names)} final features"
    )

    if fold_number == 1:
        print(
            "Historical Fold-1 target: "
            "198 numerical + 6 categorical IDs = 204."
        )

        if len(feature_names) != 204:
            print(
                "WARNING: Fold 1 did not reproduce 204 features. "
                "Do not treat this run as an exact record-feature reproduction."
            )

        print(
            f"Fold 1 numerical constants removed: "
            f"{len(dropped_numeric_cols)}"
        )

    x_train_raw = full_train.to_numpy(dtype=np.float32)
    x_valid_raw = full_valid.to_numpy(dtype=np.float32)
    x_test_raw = full_test.to_numpy(dtype=np.float32)

    # Standardize from outer-fold training rows only.
    mean = x_train_raw.mean(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    std = x_train_raw.std(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    mean = np.where(
        np.isfinite(mean),
        mean,
        0.0,
    ).astype(np.float32)

    std = np.where(
        np.isfinite(std) & (std > 1e-7),
        std,
        1.0,
    ).astype(np.float32)

    x_train = np.clip(
        (x_train_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_valid = np.clip(
        (x_valid_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_test = np.clip(
        (x_test_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    return (
        x_train,
        y_train,
        x_valid,
        y_valid,
        x_test,
        feature_names,
    )


# ## Matching network
# 
# Two hierarchical prototype-matching layers replace a conventional hidden MLP.

# ==========================================================================================
# Hierarchical prototype matching network with four similarity geometries
# ==========================================================================================

def logit_scalar(p):
    p = float(np.clip(p, 1e-5, 1 - 1e-5))
    return math.log(p / (1.0 - p))


class PrototypeMatchingLayer(nn.Module):
    """
    Prototype matching layer with a learned soft feature subset per neuron.

    Each matcher combines:
      1) Euclidean proximity
      2) cosine-direction similarity
      3) a learned signed mismatch-pattern score
      4) an M_eff-normalized weighted dot-product score
    """

    def __init__(
        self,
        input_dim,
        n_units,
        init_target_m,
        n_fire_knots=N_FIRE_KNOTS,
        pattern_hidden=PATTERN_HIDDEN,
    ):
        super().__init__()

        self.input_dim = int(input_dim)
        self.n_units = int(n_units)
        self.n_fire_knots = int(n_fire_knots)
        self.pattern_hidden = int(pattern_hidden)
        self.mask_temperature = float(MASK_TEMP_START)

        # Per-matcher log temperature multiplier.
        # alpha_i = 0 means T_i == current global mask temperature.
        self.log_temperature_scale = nn.Parameter(
            torch.randn(n_units) * LOCAL_T_LOG_SCALE_INIT_STD
        )

        self.centers = nn.Parameter(
            torch.randn(n_units, input_dim) * 0.15
        )

        self.feature_scores = nn.Parameter(
            torch.randn(n_units, input_dim) * 0.03
        )

        init_p = np.clip(
            init_target_m / max(input_dim, 1),
            1e-4,
            1 - 1e-4,
        )

        tau0 = -MASK_TEMP_START * logit_scalar(init_p)

        self.raw_tau = nn.Parameter(
            torch.full((n_units,), float(tau0))
        )

        inv_softplus_one = math.log(math.expm1(1.0))

        self.raw_fire_increments = nn.Parameter(
            torch.full(
                (n_units, 4, n_fire_knots - 1),
                inv_softplus_one,
            )
        )

        self.geometry_logits = nn.Parameter(
            torch.zeros(n_units, 4)
        )

        scale1 = 0.20 / max(input_dim, 1) ** 0.5
        scale2 = 0.20 / max(pattern_hidden, 1) ** 0.5

        self.pattern_w1 = nn.Parameter(
            torch.randn(
                n_units,
                pattern_hidden,
                input_dim,
            ) * scale1
        )

        self.pattern_b1 = nn.Parameter(
            torch.zeros(n_units, pattern_hidden)
        )

        self.pattern_w2 = nn.Parameter(
            torch.randn(n_units, pattern_hidden) * scale2
        )

        self.pattern_b2 = nn.Parameter(
            torch.zeros(n_units)
        )

    def set_mask_temperature(self, temperature):
        self.mask_temperature = float(temperature)

    def centered_log_temperature_scale(self):
        """
        Zero-center the learned log-temperature offsets within this layer.

        This removes the common/global drift direction. Individual matchers can
        still become harder or softer, but the layer as a whole remains anchored
        to the global annealing schedule.
        """
        return (
            self.log_temperature_scale
            - self.log_temperature_scale.mean()
        )

    def local_temperatures(self):
        """
        Return one relative learned mask temperature per matcher.

        T_i(t) = clip(
            T_global(t) * exp(alpha_i - mean(alpha)),
            LOCAL_T_MIN,
            LOCAL_T_MAX
        )

        Before clipping, the geometric mean of the local temperatures is exactly
        the current global temperature.
        """
        global_temperature = max(
            float(self.mask_temperature),
            1e-4,
        )

        centered_alpha = (
            self.centered_log_temperature_scale()
        )

        local_temperature = (
            global_temperature
            * torch.exp(centered_alpha)
        )

        return local_temperature.clamp(
            LOCAL_T_MIN,
            LOCAL_T_MAX,
        )

    def gates(self):
        local_temperature = (
            self.local_temperatures()
            .unsqueeze(1)
        )

        return torch.sigmoid(
            (
                self.feature_scores
                - self.raw_tau.unsqueeze(1)
            )
            / local_temperature
        )

    def fire_values(self):
        # [units, 4 geometries, knots]
        increments = (
            F.softplus(self.raw_fire_increments)
            + 1e-6
        )

        cumulative = torch.cumsum(
            increments,
            dim=2,
        )

        cumulative = (
            cumulative
            / cumulative[:, :, -1:].clamp_min(1e-8)
        )

        zero = torch.zeros(
            (self.n_units, 4, 1),
            device=cumulative.device,
            dtype=cumulative.dtype,
        )

        return torch.cat(
            [zero, cumulative],
            dim=2,
        )

    def apply_fire_curve(self, match_score):
        """
        Apply independent monotonic calibration to Euclid, cosine, pattern and dot-product.

        match_score shape: [batch, units, 4]
        """
        position = (
            match_score.clamp(0.0, 1.0)
            * (self.n_fire_knots - 1)
        )

        idx0 = (
            torch.floor(position)
            .long()
            .clamp(0, self.n_fire_knots - 2)
        )

        frac = (
            position
            - idx0.to(position.dtype)
        )

        values = self.fire_values()  # [units, 4, knots]

        values_b = values.unsqueeze(0).expand(
            match_score.shape[0],
            -1,
            -1,
            -1,
        )

        y0 = torch.gather(
            values_b,
            dim=3,
            index=idx0.unsqueeze(3),
        ).squeeze(3)

        y1 = torch.gather(
            values_b,
            dim=3,
            index=(idx0 + 1).unsqueeze(3),
        ).squeeze(3)

        return y0 + frac * (y1 - y0)

    def forward(self, x):
        weights = self.gates()
        centers = self.centers

        effective_m = (
            weights.sum(dim=1)
            .clamp_min(1e-3)
        )

        weighted_centers = weights * centers

        # Euclidean proximity.
        x2 = x.pow(2) @ weights.t()
        cross = 2.0 * (x @ weighted_centers.t())

        c2 = (
            weights * centers.pow(2)
        ).sum(dim=1).unsqueeze(0)

        distance2 = (
            (
                x2
                - cross
                + c2
            )
            / effective_m.unsqueeze(0)
        ).clamp_min(0.0)

        euclidean_match = 1.0 / (1.0 + distance2)

        # Cosine direction.
        dot = x @ weighted_centers.t()

        x_norm = torch.sqrt(
            (x.pow(2) @ weights.t()).clamp_min(1e-8)
        )

        center_norm = torch.sqrt(
            (
                weights * centers.pow(2)
            )
            .sum(dim=1)
            .clamp_min(1e-8)
        ).unsqueeze(0)

        cosine = (
            dot
            / (
                x_norm
                * center_norm
            )
        ).clamp(-1.0, 1.0)

        cosine_match = 0.5 * (cosine + 1.0)

        # Fourth geometry: M_eff-normalized weighted dot product.
        # This keeps magnitude-sensitive alignment that cosine removes,
        # while dividing by M_eff stabilizes scale across subset sizes.
        normalized_dot = (
            dot
            / effective_m.unsqueeze(0)
        )

        dot_match = torch.sigmoid(
            normalized_dot
        )

        # Learned signed mismatch pattern.
        effective_pattern_w1 = (
            self.pattern_w1
            * weights[:, None, :]
        )

        hidden = torch.einsum(
            "bd,uhd->buh",
            x,
            effective_pattern_w1,
        )

        center_offset = (
            effective_pattern_w1
            * centers[:, None, :]
        ).sum(dim=2)

        hidden = (
            hidden
            - center_offset.unsqueeze(0)
            + self.pattern_b1.unsqueeze(0)
        )

        hidden = F.gelu(hidden)

        pattern_logit = (
            hidden
            * self.pattern_w2.unsqueeze(0)
        ).sum(dim=2) + self.pattern_b2.unsqueeze(0)

        pattern_match = torch.sigmoid(
            pattern_logit
        )

        components = torch.stack(
            [
                euclidean_match,
                cosine_match,
                pattern_match,
                dot_match,
            ],
            dim=2,
        )

        # Each geometry gets its own learned monotonic response curve.
        fired_components = self.apply_fire_curve(
            components
        )

        geometry_weights = F.softmax(
            self.geometry_logits / GEOMETRY_MIX_TEMP,
            dim=1,
        )

        return (
            fired_components
            * geometry_weights.unsqueeze(0)
        ).sum(dim=2)

    def regularization(self):
        weights = self.gates()

        size_penalty = weights.mean()
        binary_penalty = (
            weights
            * (1.0 - weights)
        ).mean()

        fire = self.fire_values()

        second_difference = (
            fire[:, :, 2:]
            - 2.0 * fire[:, :, 1:-1]
            + fire[:, :, :-2]
        )

        fire_smoothness = (
            second_difference.pow(2)
            .mean()
        )

        # Penalize only RELATIVE temperature dispersion.
        # A common shift is removed by construction and therefore cannot
        # override the global annealing schedule.
        centered_alpha = (
            self.centered_log_temperature_scale()
        )

        local_t_penalty = (
            centered_alpha.pow(2)
            .mean()
        )

        # Encourage different matchers to use somewhat different feature subsets.
        # This is deliberately weak: overlap is allowed, exact duplication is discouraged.
        normalized_gates = (
            weights
            / weights.norm(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

        gate_similarity = (
            normalized_gates
            @ normalized_gates.t()
        )

        n = gate_similarity.shape[0]

        if n > 1:
            eye = torch.eye(
                n,
                device=gate_similarity.device,
                dtype=gate_similarity.dtype,
            )

            gate_diversity = (
                (
                    gate_similarity
                    * (1.0 - eye)
                ).pow(2).sum()
                / (n * (n - 1))
            )
        else:
            gate_diversity = torch.zeros(
                (),
                device=weights.device,
                dtype=weights.dtype,
            )

        return (
            LAMBDA_MASK_SIZE * size_penalty
            + LAMBDA_MASK_BINARY * binary_penalty
            + LAMBDA_FIRE_SMOOTH * fire_smoothness
            + LAMBDA_LOCAL_T * local_t_penalty
            + LAMBDA_GATE_DIVERSITY * gate_diversity
        )

    @torch.no_grad()
    def initialize_centers_from_rows(self, rows, seed):
        rng = np.random.default_rng(seed)
        n_rows = rows.shape[0]

        idx = rng.choice(
            n_rows,
            size=self.n_units,
            replace=(n_rows < self.n_units),
        )

        chosen = rows[idx]

        if not torch.is_tensor(chosen):
            chosen = torch.as_tensor(
                chosen,
                dtype=self.centers.dtype,
                device=self.centers.device,
            )
        else:
            chosen = chosen.to(
                self.centers.device,
                dtype=self.centers.dtype,
            )

        self.centers.copy_(chosen)

    @torch.no_grad()
    def diagnostics(self):
        weights = self.gates()
        effective_m = weights.sum(dim=1)

        geometry = F.softmax(
            self.geometry_logits,
            dim=1,
        )

        local_t = self.local_temperatures()
        return {
            "m_mean": float(effective_m.mean().cpu()),
            "geometry_mean": geometry.mean(dim=0).cpu().numpy(),
            "t_global": float(self.mask_temperature),
            "t_mean": float(local_t.mean().cpu()),
            "t_min": float(local_t.min().cpu()),
            "t_max": float(local_t.max().cpu()),
        }


class HierarchicalMatcher(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.match1 = PrototypeMatchingLayer(
            input_dim=input_dim,
            n_units=N_MATCH1,
            init_target_m=INIT_TARGET_M1,
        )

        self.match2 = PrototypeMatchingLayer(
            input_dim=N_MATCH1,
            n_units=N_MATCH2,
            init_target_m=INIT_TARGET_M2,
        )

        # ------------------------------------------------------------------
        # Sample-dependent soft routing from h1 -> h2
        #
        # h1 first creates broad/local pattern evidence. A tiny low-rank
        # contextual router then decides which h1 responses should be
        # emphasized or suppressed for this particular sample before h2
        # performs another prototype-matching stage.
        #
        # The final projection is zero-initialized, therefore:
        #     sigmoid(0) * 2 = 1
        # and the model starts exactly as the non-routed baseline.
        # ------------------------------------------------------------------
        self.route_down = nn.Linear(
            N_MATCH1,
            ROUTING_RANK,
            bias=True,
        )

        self.route_up = nn.Linear(
            ROUTING_RANK,
            N_MATCH1,
            bias=True,
        )

        nn.init.normal_(
            self.route_down.weight,
            mean=0.0,
            std=0.02 / max(N_MATCH1, 1) ** 0.5,
        )
        nn.init.zeros_(self.route_down.bias)

        nn.init.zeros_(self.route_up.weight)
        nn.init.zeros_(self.route_up.bias)

        # Linear skip readout from both hierarchy levels.
        #
        # The model can use:
        #   - direct first-layer pattern evidence (h1)
        #   - higher-order second-layer pattern evidence (h2)
        #
        # without forcing all useful information through the second matcher layer.
        self.head = nn.Linear(
            N_MATCH1 + N_MATCH2,
            1,
        )

    def set_mask_temperature(self, temperature):
        self.match1.set_mask_temperature(temperature)
        self.match2.set_mask_temperature(temperature)

    def routing_weights(self, h1):
        """
        Sample-dependent multiplicative routing over h1.

        Returns positive weights with per-sample mean ~= 1.0.
        At initialization every route is exactly 1.0.
        """
        context = torch.tanh(
            self.route_down(h1)
        )

        route_logits = self.route_up(
            context
        )

        route = 2.0 * torch.sigmoid(
            route_logits
        )

        # Remove the trivial global-scale degree of freedom.
        route = (
            route
            / route.mean(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-6)
        )

        return route

    def forward(self, x):
        h1 = self.match1(x)

        route = self.routing_weights(
            h1
        )

        routed_h1 = (
            h1 * route
        )

        h2 = self.match2(
            routed_h1
        )

        # Keep the direct h1 path untouched. Only the deeper matching path
        # receives the conditionally routed representation.
        readout = torch.cat(
            [h1, h2],
            dim=1,
        )

        return self.head(readout).squeeze(1)

    def regularization(self):
        # Weak regularization keeps the contextual router near identity unless
        # the predictive loss benefits from sample-specific routing.
        routing_penalty = (
            self.route_up.weight.pow(2).mean()
            + self.route_up.bias.pow(2).mean()
        )

        return (
            self.match1.regularization()
            + self.match2.regularization()
            + LAMBDA_HEAD_L2
            * self.head.weight.pow(2).mean()
            + LAMBDA_ROUTING
            * routing_penalty
        )


    @torch.no_grad()
    def initialize_centers(self, x_train, seed):
        rng = np.random.default_rng(seed)

        n_rows = min(
            CENTER_INIT_ROWS,
            len(x_train),
        )

        idx = rng.choice(
            len(x_train),
            size=n_rows,
            replace=False,
        )

        rows = torch.from_numpy(
            x_train[idx]
        ).to(DEVICE)

        self.match1.initialize_centers_from_rows(
            rows,
            seed + 1,
        )

        layer1_parts = []

        for start in range(
            0,
            len(rows),
            2048,
        ):
            layer1_parts.append(
                self.match1(
                    rows[start:start + 2048]
                ).detach()
            )

        layer1_rows = torch.cat(
            layer1_parts,
            dim=0,
        )

        self.match2.initialize_centers_from_rows(
            layer1_rows,
            seed + 2,
        )


class BinaryDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return (
            torch.from_numpy(self.x[index]),
            torch.tensor(
                self.y[index],
                dtype=torch.float32,
            ),
        )


@torch.no_grad()
def predict_logits(model, x, batch_size=PRED_BATCH_SIZE):
    model.eval()

    output = np.empty(
        len(x),
        dtype=np.float32,
    )

    for start in range(
        0,
        len(x),
        batch_size,
    ):
        end = min(
            start + batch_size,
            len(x),
        )

        xb = torch.from_numpy(
            x[start:end]
        ).to(
            DEVICE,
            non_blocking=True,
        )

        output[start:end] = (
            model(xb)
            .float()
            .cpu()
            .numpy()
        )

    return output


def sigmoid_np(x):
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


# ## Training
# 
# Each fold uses early stopping on validation ROC AUC and saves its best checkpoint.

# ==========================================================================================
# Train one outer fold
# ==========================================================================================

def train_one_fold(
    fold_number,
    train_idx,
    valid_idx,
    train_raw,
    test_raw,
):
    print()
    print("=" * 120)
    print(
        f"FOLD {fold_number}/{N_SPLITS} | "
        f"train={len(train_idx):,} | "
        f"valid={len(valid_idx):,}"
    )
    print("=" * 120)

    (
        x_train,
        y_train,
        x_valid,
        y_valid,
        x_test,
        feature_names,
    ) = build_fold_features(
        train_raw,
        test_raw,
        train_idx,
        valid_idx,
        fold_number,
    )

    # Use a deterministic but distinct seed for each outer fold.
    model_seed = SEED + (fold_number - 1)
    center_seed = model_seed + 8100

    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(model_seed)
        torch.cuda.manual_seed_all(model_seed)

    dataset = BinaryDataset(
        x_train,
        y_train,
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

    model = HierarchicalMatcher(
        input_dim=x_train.shape[1]
    ).to(DEVICE)

    model.initialize_centers(
        x_train,
        seed=center_seed,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=MAX_EPOCHS,
        eta_min=LR * 0.12,
    )

    criterion = nn.BCEWithLogitsLoss()

    history = []
    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    best_mask_temperature = None
    epochs_without_improvement = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        fraction = (
            (epoch - 1)
            / max(MAX_EPOCHS - 1, 1)
        )

        mask_temperature = (
            MASK_TEMP_START
            * (
                MASK_TEMP_END
                / MASK_TEMP_START
            ) ** fraction
        )

        model.set_mask_temperature(
            mask_temperature
        )

        model.train()

        loss_sum = 0.0
        n_seen = 0

        for xb, yb in loader:
            xb = xb.to(
                DEVICE,
                non_blocking=True,
            )

            yb = yb.to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(xb)

            data_loss = criterion(
                logits,
                yb,
            )

            loss = (
                data_loss
                + model.regularization()
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            batch_size = len(xb)

            loss_sum += (
                float(data_loss.detach().cpu())
                * batch_size
            )

            n_seen += batch_size

        scheduler.step()

        # Only validation AUC is computed every epoch.
        # Skipping full-train inference saves substantial time in 5-fold CPU runs.
        valid_logits = predict_logits(
            model,
            x_valid,
        )

        valid_auc = roc_auc_score(
            y_valid,
            valid_logits,
        )

        diag1 = model.match1.diagnostics()
        diag2 = model.match2.diagnostics()

        learning_rate = optimizer.param_groups[0]["lr"]

        row = {
            "fold": fold_number,
            "epoch": epoch,
            "loss": loss_sum / max(n_seen, 1),
            "valid_auc": valid_auc,
            "learning_rate": learning_rate,
            "mask_temperature": mask_temperature,
            "m1_mean": diag1["m_mean"],
            "m2_mean": diag2["m_mean"],
            "t1_mean": diag1["t_mean"],
            "t1_min": diag1["t_min"],
            "t1_max": diag1["t_max"],
            "t2_mean": diag2["t_mean"],
            "t2_min": diag2["t_min"],
            "t2_max": diag2["t_max"],
            "geo1_e": diag1["geometry_mean"][0],
            "geo1_c": diag1["geometry_mean"][1],
            "geo1_p": diag1["geometry_mean"][2],
            "geo1_d": diag1["geometry_mean"][3],
            "geo2_e": diag2["geometry_mean"][0],
            "geo2_c": diag2["geometry_mean"][1],
            "geo2_p": diag2["geometry_mean"][2],
            "geo2_d": diag2["geometry_mean"][3],
        }

        history.append(row)

        improved = (
            valid_auc
            > best_auc + 1e-7
        )

        if improved:
            best_auc = valid_auc
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )

            # mask_temperature is a plain Python attribute and is not
            # included in state_dict(), so save it with the best checkpoint.
            best_mask_temperature = float(mask_temperature)

            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        marker = " *BEST*" if improved else ""

        print(
            f"Fold {fold_number} | "
            f"epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"loss={row['loss']:.6f} | "
            f"ValAUC={valid_auc:.7f}{marker} | "
            f"lr={learning_rate:.2e} | "
            f"M={diag1['m_mean']:.1f}->{diag2['m_mean']:.1f} | "
            f"Tg={diag1['t_global']:.3f} | "
            f"T1={diag1['t_mean']:.3f}"
            f"[{diag1['t_min']:.3f},{diag1['t_max']:.3f}] | "
            f"T2={diag2['t_mean']:.3f}"
            f"[{diag2['t_min']:.3f},{diag2['t_max']:.3f}] | "
            f"geo1(E/C/P/D)={diag1['geometry_mean'][0]:.2f}/"
            f"{diag1['geometry_mean'][1]:.2f}/"
            f"{diag1['geometry_mean'][2]:.2f}/"
            f"{diag1['geometry_mean'][3]:.2f} | "
            f"geo2(E/C/P/D)={diag2['geometry_mean'][0]:.2f}/"
            f"{diag2['geometry_mean'][1]:.2f}/"
            f"{diag2['geometry_mean'][2]:.2f}/"
            f"{diag2['geometry_mean'][3]:.2f}"
        )

        pd.DataFrame(history).to_csv(
            os.path.join(
                OUTPUT_DIR,
                f"hpmn_final_fold{fold_number}_history.csv",
            ),
            index=False,
        )

        if (
            epochs_without_improvement
            >= PATIENCE
        ):
            print(
                f"Fold {fold_number}: "
                f"early stopping at epoch {epoch}."
            )
            break

    if best_state is None:
        raise RuntimeError(
            f"Fold {fold_number}: no checkpoint was created."
        )

    model.load_state_dict(
        best_state
    )

    # Restore the non-state_dict mask temperature from the best epoch.
    model.set_mask_temperature(
        best_mask_temperature
    )

    model.eval()

    valid_logits = predict_logits(
        model,
        x_valid,
    )

    test_logits = predict_logits(
        model,
        x_test,
    )

    valid_prob = sigmoid_np(
        valid_logits
    )

    test_prob = sigmoid_np(
        test_logits
    )

    final_auc = roc_auc_score(
        y_valid,
        valid_prob,
    )

    print(
        f"Fold {fold_number} complete | "
        f"best epoch={best_epoch} | "
        f"AUC={final_auc:.7f}"
    )

    # Save fold-level validation predictions.
    pd.DataFrame({
        "row_index": valid_idx,
        TARGET: y_valid,
        "prediction": valid_prob,
    }).to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"hpmn_final_fold{fold_number}_validation.csv",
        ),
        index=False,
    )

    # Save a reproducible model checkpoint.
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "fold": fold_number,
            "best_epoch": best_epoch,
            "best_auc": final_auc,
            "best_mask_temperature": best_mask_temperature,
            "model_seed": model_seed,
            "center_seed": center_seed,
            "feature_names": feature_names,
            "n_features": len(feature_names),
            "n_match1": N_MATCH1,
            "n_match2": N_MATCH2,
            "pattern_hidden": PATTERN_HIDDEN,
            "n_fire_knots": N_FIRE_KNOTS,
            "init_target_m1": INIT_TARGET_M1,
            "init_target_m2": INIT_TARGET_M2,
            "local_t_min": LOCAL_T_MIN,
            "local_t_max": LOCAL_T_MAX,
            "routing_rank": ROUTING_RANK,
        },
        os.path.join(
            OUTPUT_DIR,
            f"hpmn_final_fold{fold_number}_model.pt",
        ),
    )

    result = {
        "fold": fold_number,
        "auc": final_auc,
        "best_epoch": best_epoch,
        "n_features": len(feature_names),
    }

    # Free the large fold-specific matrices before moving to the next fold.
    del (
        dataset,
        loader,
        model,
        optimizer,
        scheduler,
        x_train,
        x_valid,
        x_test,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        result,
        valid_idx,
        valid_prob,
        test_prob,
    )

# ==========================================================================================
# Five-fold training and Kaggle submission
# ==========================================================================================

outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

oof_pred = np.zeros(len(train), dtype=np.float32)
test_fold_pred = []
fold_results = []

for fold_number, (train_idx, valid_idx) in enumerate(
    outer_cv.split(np.zeros(len(train)), y),
    start=1,
):
    result, fold_valid_idx, fold_valid_prob, fold_test_prob = train_one_fold(
        fold_number=fold_number,
        train_idx=train_idx,
        valid_idx=valid_idx,
        train_raw=train,
        test_raw=test,
    )

    oof_pred[fold_valid_idx] = fold_valid_prob
    test_fold_pred.append(fold_test_prob.astype(np.float32, copy=False))
    fold_results.append(result)

    # Save fold-level progress immediately; useful for long CPU runs.
    pd.DataFrame(fold_results).to_csv(
        os.path.join(OUTPUT_DIR, "hpmn_fold_summary.csv"),
        index=False,
    )

# Overall out-of-fold score across all five validation folds.
oof_auc = roc_auc_score(y, oof_pred)

# Final Kaggle prediction = arithmetic mean of the five fold probabilities.
test_pred = np.mean(
    np.stack(test_fold_pred, axis=0),
    axis=0,
).astype(np.float32)

pd.DataFrame({
    "row_index": np.arange(len(train)),
    TARGET: y,
    "prediction": oof_pred,
}).to_csv(
    os.path.join(OUTPUT_DIR, "hpmn_oof_predictions.csv"),
    index=False,
)

summary = pd.DataFrame(fold_results)
summary.to_csv(
    os.path.join(OUTPUT_DIR, "hpmn_fold_summary.csv"),
    index=False,
)

if sample_path is not None and os.path.exists(sample_path):
    submission = pd.read_csv(sample_path)

    if TARGET not in submission.columns:
        raise ValueError(
            f"Expected target column '{TARGET}' in sample_submission.csv; "
            f"found {submission.columns.tolist()}"
        )

    if len(submission) != len(test_pred):
        raise ValueError(
            "sample_submission.csv and test predictions have different row counts."
        )

    submission[TARGET] = test_pred
else:
    submission = pd.DataFrame({TARGET: test_pred})

    if ID in test.columns:
        submission.insert(0, ID, test[ID].to_numpy())

submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print()
print("=" * 120)
print("FINAL 5-FOLD HPMN RESULTS")
print("=" * 120)

for result in fold_results:
    print(
        f"Fold {int(result['fold'])}: "
        f"AUC={result['auc']:.7f} | "
        f"best epoch={int(result['best_epoch'])} | "
        f"features={int(result['n_features'])}"
    )

print(f"Mean fold AUC : {np.mean([r['auc'] for r in fold_results]):.7f}")
print(f"OOF AUC       : {oof_auc:.7f}")
print(f"Submission    : {submission_path}")
print("Test blending : arithmetic mean of 5 fold probabilities")
print("=" * 120)
